# Phase 6 — which network, measured rather than assumed

Four training runs that differ in the network and in nothing else. The baseline
is a U-Net with a ResNet-34 encoder because that is where the project started,
not because anything measured said so, and this notebook is what replaces the
assumption with a number.

| run | decoder | encoder | what it isolates |
|---|---|---|---|
| a | Unet | resnet34 | the reference, at this epoch count |
| b | Unet | tu-convnext_tiny | the encoder alone |
| c | UPerNet | tu-convnext_tiny | the decoder, on b's encoder |
| d | Segformer | mit_b2 | a different family |

Run *d* trains on batches of two where the others use four. A transformer
compares every patch with every other, so at 1024 pixels a batch of four does
not fit a T4 and this run was lost to that on the first attempt. Its number is
therefore not on quite the same footing as the other three, and saying so is
part of quoting it.

**Second round (e, f, g).** All four lost to the reference, and the reason is
measured: ConvNeXt and MiT expose no feature map at half resolution, which at
1024 is where a filament a few pixels wide lives. Three more runs, each changing
one thing against *a*:

| run | changes | from *a* | what it isolates |
|---|---|---|---|
| g | the training target | union of every annotator of the frame | how the network is taught, not what it is |
| f | the resolution | 2048 at batch 1 (same pixels per step) | whether the missing level is the problem |
| e | the encoder | tu-hrnet_w32 at batch 2 | a network that keeps a half-resolution branch throughout |

Run *g* changes only what the training side is asked for; validation and PQ are
still measured against each annotator's own tracing, so its number sits beside
a to d. Run *f* is scored with the rejoining distances doubled, because they are
counted in pixels of the probability map and its map is twice as fine.

The two T4s train two runs at once, *f* on one and *g* then *e* on the other,
because every hour of this notebook is an hour of the weekly GPU quota whether
the second card is busy or not.

Why these and not others. Filaments are thin, they cover 0.35% of a frame, and
the ones missing from the probability map entirely are the small ones: a
correct box drawn around 266 of fold 0's filaments contains no pixel above 0.05.
An encoder-decoder that halves its resolution five times has the most to lose
there. UPerNet pools context over the whole frame before decoding; the
Segformer encoder attends globally from its first stage. Both are reasons about
this data, not about publication dates.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2**
2. Internet: **on**
3. Add the competition data as an input
4. **Save Version** with *Save output* on, or the checkpoints and maps are discarded

Expected wall clock, first round, on one T4: roughly 50 minutes for run a, 60
to 90 for each of the others, so four to five hours in total, plus twenty
minutes writing probability maps. Second round, with a to d restored from an
earlier version: three to four hours, the longest queue being *f*.

**Re-running after one of them failed.** A run whose `best.pt` is already in
`/kaggle/working` is scored from that file instead of being trained again, so
recovering a single crashed run costs only that run. Starting from a fresh
session, attach the earlier version's output as an input and copy the finished
directories into `/kaggle/working` first; otherwise every run trains again.

## 1. Clone the repository and put it on the path

The repository is cloned rather than installed, because `configs/paths.yaml`
and the frozen splits in `configs/splits/` sit beside the package rather than
inside it. Set `REF` to the branch or commit this run is to be reproducible
from.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase6-round2"  # branch or commit hash
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())
if torch.cuda.is_available():
    print("name:", torch.cuda.get_device_name(0))

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

dataset_root = candidates[0].parent.parent
os.environ["MAGFILO_ROOT"] = str(dataset_root)
print("MAGFILO_ROOT =", dataset_root)

paths = load_paths().require_dataset()
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))

## 3. The four runs

The ground truth is encoded once and handed to every scoring call: it takes as
long as the inference and does not depend on the model.

Scoring uses the post-processing Phase 3 settled on — threshold 0.5, minimum
area 400, rejoining within 24 pixels — for all four runs. The defaults in the
code are not that configuration, so they are named explicitly here. Two runs
are comparable only when the chain behind them is the same.

In [ ]:
from dataclasses import replace

from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.postprocess.join import DEFAULT_MAX_OFFSET
from filament.training.config import TrainConfig

FIRST_ROUND = ["a_unet_resnet34", "b_unet_convnext", "c_upernet_convnext", "d_segformer_mit"]
SECOND_ROUND = ["g_unet_resnet34_union", "f_unet_resnet34_2048", "e_unet_hrnet"]
RUNS = FIRST_ROUND + SECOND_ROUND

# The configuration Phase 3 settled on, applied identically to every run.
SCORING = {"threshold": 0.5, "min_area": 400, "join_gap": 24.0}
# The resolution the rejoining distances above were chosen at.
SCORING_SIZE = 1024


def scoring_for(config):
    """SCORING, expressed in pixels of this run's probability map.

    The rejoining distances are counted in pixels of the map, so a run whose
    map is twice as fine needs them doubled to mean the same distance on the
    Sun. The minimum area is in pixels of the full frame and never changes.
    At 1024 this is exactly SCORING plus the default axis offset.
    """
    scale = config.image_size / SCORING_SIZE
    return SCORING | {
        "join_gap": SCORING["join_gap"] * scale,
        "join_offset": DEFAULT_MAX_OFFSET * scale,
    }


dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(0, f"{CHECKOUT}/configs/splits").val
print(f"fold 0: {len(val_stems)} validation frames")

configs = {}
for name in RUNS:
    configs[name] = replace(
        TrainConfig.from_yaml(f"{CHECKOUT}/configs/phase6/{name}.yaml"),
        num_workers=2,
        output_dir=Path(f"/kaggle/working/{name}"),
    )
    config = configs[name]
    print(
        f"{name:<24} {config.architecture:<10} {config.encoder:<16} "
        f"{config.image_size}px batch {config.batch_size} "
        f"union {config.union_targets}  scoring {scoring_for(config)}"
    )

In [ ]:
import shutil

from filament.submit.rle import masks_to_gt_df, read_submission, write_submission

# Whatever an earlier version finished is carried forward, so that adding a
# run to the comparison costs that run and not the ones already done.
for name in [f"sweep_{run}.csv" for run in RUNS] + ["phase6_summary.json"]:
    destination = Path("/kaggle/working") / name
    if not destination.exists():
        found = sorted(Path("/kaggle/input").glob(f"**/{name}"))
        if found:
            shutil.copy(found[0], destination)
            print(f"{name}: restored")

gt_path = Path("/kaggle/working/gt_fold0.csv")
if not gt_path.exists():
    # An earlier version of this notebook, attached as an input, has it already.
    found = sorted(Path("/kaggle/input").glob("**/gt_fold0.csv"))
    if found:
        shutil.copy(found[0], gt_path)
        print(f"ground truth restored from {found[0]}")

if gt_path.exists():
    gt_df = read_submission(gt_path)
else:
    gt_df = masks_to_gt_df(dataset, val_stems)
    write_submission(gt_df, gt_path)
print(f"{len(gt_df)} ground-truth filaments")

In [ ]:
import json
import logging
import time

import numpy as np

from filament.data.image import load_grayscale
from filament.evaluation import evaluate, predict_probability
from filament.training.loop import HISTORY_NAME, load_checkpoint, train

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Everything here runs on whatever is attached. With the earlier version's work
# among the inputs there is nothing left to train, and the accelerator can be
# turned off; without it, this needs a GPU to finish in hours rather than days.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

EVAL_NAME = "eval_fold0.json"


def training_record(output_dir):
    """Minutes and best validation loss, from the history a run left behind."""
    history_path = output_dir / HISTORY_NAME
    if not history_path.exists():
        return None, None
    history = json.loads(history_path.read_text())
    minutes = round(sum(item["seconds"] for item in history) / 60, 1)
    return minutes, round(min(item["val_loss"] for item in history), 4)


def write_maps(model, name):
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
    maps_dir.mkdir(parents=True, exist_ok=True)
    if len(list(maps_dir.glob("*.npy"))) < len(val_stems):
        for stem in val_stems:
            probability = predict_probability(
                model,
                load_grayscale(paths.train_images / f"{stem}.jpeg"),
                size=configs[name].image_size,
                device=DEVICE,
            )
            np.save(maps_dir / f"{stem}.npy", probability.astype(np.float16))
    written = sorted(maps_dir.glob("*.npy"))
    print(f"{name}: {len(written)} maps, {sum(i.stat().st_size for i in written) / 1e6:.0f} MB")


def train_and_score(name):
    # Train one configuration if it is not already trained, then score it and
    # write its probability maps.
    config = configs[name]
    checkpoint = config.output_dir / "best.pt"
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")

    # Scored by an earlier version with its maps in place: scoring again would
    # spend a few GPU minutes per run to print the same numbers.
    eval_path = config.output_dir / EVAL_NAME
    if eval_path.exists() and len(list(maps_dir.glob("*.npy"))) >= len(val_stems):
        summary = json.loads(eval_path.read_text())
        print(f"{name}: already scored, PQ {summary['pq']}")
        return summary

    # Already trained, here or by the parallel step above: a session that died
    # part-way should not cost the runs that finished. Delete the directory to
    # force a retrain.
    if checkpoint.exists():
        print(f"{name}: reusing the checkpoint already in {checkpoint.parent}")
        training_minutes, best_val_loss = training_record(config.output_dir)
        best_epoch = torch.load(checkpoint, map_location="cpu", weights_only=False)["epoch"]
    else:
        started = time.perf_counter()
        outcome = train(config)
        training_minutes = round((time.perf_counter() - started) / 60, 1)
        best_epoch = outcome.best_epoch
        best_val_loss = round(outcome.best_val_loss, 4)
        checkpoint = outcome.checkpoint

    model, _ = load_checkpoint(checkpoint)
    scoring = scoring_for(config)
    evaluation, _ = evaluate(
        model,
        dataset,
        paths.train_images,
        val_stems,
        size=config.image_size,
        device=DEVICE,
        gt_df=gt_df,
        **scoring,
    )
    summary = evaluation.to_dict() | {
        "architecture": config.architecture,
        "encoder": config.encoder,
        "image_size": config.image_size,
        "batch_size": config.batch_size,
        "union_targets": config.union_targets,
        "scoring": scoring,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "training_minutes": training_minutes,
    }
    eval_path.write_text(json.dumps(summary, indent=2))
    print(evaluation, f"| {training_minutes} min")

    # Written now rather than after every run: if the session ends early, the
    # runs that finished still leave behind what a sweep needs.
    write_maps(model, name)

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return summary


# Kaggle empties /kaggle/working when a session starts, so an earlier version's
# work is only reachable through the inputs. Copying it back is what lets Run All
# finish a partly-done comparison rather than repeating the hours that succeeded.
for folder in list(RUNS) + [f"prob_fold0_{name}" for name in RUNS]:
    destination = Path("/kaggle/working") / folder
    if destination.exists() and any(destination.iterdir()):
        continue
    found = [item for item in Path("/kaggle/input").glob(f"**/{folder}") if item.is_dir()]
    if found:
        shutil.copytree(found[0], destination, dirs_exist_ok=True)
        print(f"{folder}: restored from {found[0]}")

### Training two runs at once

Each untrained run of the second round is handed to `scripts/train.py` in its
own process, pinned to one card with `CUDA_VISIBLE_DEVICES`. Processes rather
than threads: each seeds its own random state, so a run's initial weights do
not depend on what the other card happened to draw first.

With one card, or none, the queues collapse into one and run in turn. A run
that fails here is not retried here; the scoring loop below finds no
checkpoint for it and trains it in-process, so the notebook still reaches its
end.

In [ ]:
import os
import subprocess
import threading

gpu_count = torch.cuda.device_count()
# f is the longest single run, so it gets a card to itself.
if gpu_count >= 2:
    QUEUES = {0: ["f_unet_resnet34_2048"], 1: ["g_unet_resnet34_union", "e_unet_hrnet"]}
else:
    QUEUES = {0: ["g_unet_resnet34_union", "f_unet_resnet34_2048", "e_unet_hrnet"]}
print(f"{gpu_count} GPU(s):", QUEUES)

print_lock = threading.Lock()
failures = {}


def run_queue(gpu, names):
    for name in names:
        config = configs[name]
        if (config.output_dir / "best.pt").exists():
            with print_lock:
                print(f"[gpu{gpu}] {name}: already trained, skipped")
            continue
        config.output_dir.mkdir(parents=True, exist_ok=True)
        command = [
            sys.executable,
            f"{CHECKOUT}/scripts/train.py",
            "--config",
            f"{CHECKOUT}/configs/phase6/{name}.yaml",
            "--output-dir",
            str(config.output_dir),
            "--num-workers",
            str(config.num_workers),
        ]
        environment = os.environ | {
            "CUDA_VISIBLE_DEVICES": str(gpu),
            "PYTHONPATH": f"{CHECKOUT}/src",
        }
        with print_lock:
            print(f"[gpu{gpu}] {name}: started")
        started = time.perf_counter()
        # The log is kept beside the checkpoint, so it survives in the output
        # even if this cell's printout is truncated.
        with open(config.output_dir / "train.log", "w") as log:
            process = subprocess.Popen(
                command,
                cwd=CHECKOUT,
                env=environment,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in process.stdout:
                log.write(line)
                log.flush()
                if "Epoch" in line or "Error" in line or "Traceback" in line:
                    with print_lock:
                        print(f"[gpu{gpu}] {name}: {line.rstrip()}")
            exit_code = process.wait()
        minutes = (time.perf_counter() - started) / 60
        with print_lock:
            print(f"[gpu{gpu}] {name}: exit {exit_code} after {minutes:.1f} min")
        if exit_code != 0:
            failures[name] = exit_code


threads = [
    threading.Thread(target=run_queue, args=(gpu, names), daemon=True)
    for gpu, names in QUEUES.items()
]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

for name, exit_code in failures.items():
    log_path = configs[name].output_dir / "train.log"
    print(f"{name} FAILED with exit code {exit_code}; last lines of {log_path}:")
    print("".join(log_path.read_text().splitlines(keepends=True)[-20:]))

In [ ]:
results = {}
for name in RUNS:
    print("=" * 70, name, "=" * 70)
    try:
        results[name] = train_and_score(name)
    except Exception as error:
        # A run that will not fit, or will not build, must not take the others
        # with it. There is no second attempt at this session, so the notebook
        # reaches its end even when one of them does not.
        print(f"{name}: FAILED -- {type(error).__name__}: {error}")
        results[name] = {"failed": f"{type(error).__name__}: {error}"}
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

Path("/kaggle/working/phase6_summary.json").write_text(json.dumps(results, indent=2))

## 4. The comparison

The incumbent is **PQ 0.3756**, which is this same network trained for fifteen
epochs and scored through the post-processing Phase 3 tuned. Run *a* is that
network at twenty, so the gap between the two is the epoch count alone and says
how much of any difference below is length rather than architecture.

Adoption needs **+0.01 over run a** and an explanation of why it moved. SQ and
RQ are shown apart because they say different things: SQ is how well a matched
filament is drawn, RQ is how many were matched at all. Nothing tried so far has
moved SQ out of 0.645 to 0.665, and whether any of these does is the question
this notebook exists to answer.

For the second round the question is narrower: does changing one thing against
*a* move it by +0.01, and can the movement be explained. Run *c* scored 0.3309
and 0.3256 on the same seed, so a difference below about 0.005 is within what a
rerun alone can produce.

In [ ]:
import pandas as pd

# Only the runs that produced a score: a failed one carries an explanation
# rather than the columns below, and asking for them would end the notebook
# here, after the training it exists to report.
# Runs scored before image_size and union_targets were recorded are 1024 and
# per-annotator, which is what the defaults fill in.
scored = {
    name: {"image_size": 1024, "union_targets": False} | row
    for name, row in results.items()
    if "pq" in row
}
for name, row in results.items():
    if "pq" not in row:
        print(f"{name}: {row.get('failed', 'no result')}")

table = pd.DataFrame(scored).T[
    [
        "architecture",
        "encoder",
        "image_size",
        "batch_size",
        "union_targets",
        "pq",
        "sq",
        "rq",
        "tp",
        "fp",
        "fn",
        "fused",
        "split",
        "best_epoch",
        "training_minutes",
    ]
]
# A frame built from dictionaries holding both text and numbers comes back as
# text throughout, so subtracting one column from another fails. The numeric
# columns are converted back before they are used.
NUMERIC = [
    "image_size",
    "batch_size",
    "pq",
    "sq",
    "rq",
    "tp",
    "fp",
    "fn",
    "fused",
    "split",
    "best_epoch",
    "training_minutes",
]
table[NUMERIC] = table[NUMERIC].apply(pd.to_numeric, errors="coerce")

if "a_unet_resnet34" in table.index:
    table["pq_vs_a"] = (table["pq"] - table.loc["a_unet_resnet34", "pq"]).round(4)
table

## 5. Any maps still missing

Section 3 writes each run's maps as soon as that run is trained, so by now
they normally exist. This fills a gap left by a checkpoint that was reused from
an earlier session without its maps, and does nothing otherwise.

The maps are what the sweep below reads, and what any later sweep reads without
a GPU. Float16 keeps one run's fold to about 300 MB.

In [ ]:
for name in RUNS:
    checkpoint = Path(f"/kaggle/working/{name}/best.pt")
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
    if not checkpoint.exists():
        print(f"{name}: no checkpoint, skipped")
        continue
    if len(list(maps_dir.glob("*.npy"))) == len(val_stems):
        print(f"{name}: maps already written")
        continue

    model, _ = load_checkpoint(checkpoint)
    maps_dir.mkdir(parents=True, exist_ok=True)
    for stem in val_stems:
        probability = predict_probability(
            model,
            load_grayscale(paths.train_images / f"{stem}.jpeg"),
            size=configs[name].image_size,
            device=DEVICE,
        )
        np.save(maps_dir / f"{stem}.npy", probability.astype(np.float16))
    print(f"{name}: {len(list(maps_dir.glob('*.npy')))} maps written")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6. Each network at its own post-processing

Section 3 scored every run through one chain, which is what makes the networks
comparable. It is not what decides whether a network is any good: the
threshold, the minimum area and the rejoining distance were tuned on the
incumbent U-Net, and a network whose probabilities sit elsewhere is judged by
a gate that was not built for it. The two failures seen so far point opposite
ways, one emitting a hundred predictions too many and another ninety true ones
too few, which is what a mismatched threshold looks like as much as it is what
a worse network looks like.

So both readings are kept, as Phase 5 kept them: at the shared setting, and at
each network's own best. **The adoption decision is made on the shared
setting.** The retuned figure says whether a rejected network was rejected for
the right reason.

Nothing here needs a GPU. It reads the maps written above.

**One check before the numbers are used.** The sweep and the scoring in section
3 are two different code paths, and they do not treat the edge of the solar
disk identically: scoring shrinks the 16-pixel margin to match the half-size
map, while the sweep does not. A prediction sitting just past the limb can
therefore survive one path and not the other. If the two disagree at the same
setting, a difference between them measures the path rather than the setting,
and the retuned figures are not usable until that is settled. The cell prints
both so the question is answered rather than assumed.

**Only the first round is swept here.** The sweep costs about twenty minutes a
run, and a GPU session charges for them whether the card is used or not. The
second round's maps are saved and can be swept later in a session with the
accelerator off. Run *f*'s maps are at 2048, so its disks would need scaling by
1 rather than 0.5.

In [ ]:
import pandas as pd

from filament.data.disk import detect_disk
from filament.postprocess.search import grid, load_maps, sweep

# The saved maps are the raw model output rather than a disk-masked one, so the
# sweep has to be given the disk. It is the same disk for every configuration,
# so it is found once and reused.
disks = {
    stem: detect_disk(load_grayscale(paths.train_images / f"{stem}.jpeg")).scaled(0.5)
    for stem in val_stems
}

# Threshold is the axis worth the most here. Phase 3 found it inert on the
# incumbent -- every value from 0.3 to 0.7 within 0.001 -- but that is a
# property of one network's probabilities, not of the problem.
SETTINGS = grid(
    threshold=[0.3, 0.4, 0.5, 0.6, 0.7],
    min_area=[200, 400, 600],
    join_gap=[0.0, 24.0],
)
print(f"{len(SETTINGS)} settings per run")

SWEEP_RUNS = FIRST_ROUND

retuned = {}
for name in SWEEP_RUNS:
    maps_dir = Path(f"/kaggle/working/prob_fold0_{name}")
    if not maps_dir.exists():
        print(f"{name}: no maps, skipped")
        continue

    table_path = Path(f"/kaggle/working/sweep_{name}.csv")
    if table_path.exists():
        # Swept by an earlier version. The file is written best first.
        done = pd.read_csv(table_path)
        row = done.iloc[0]
        retuned[name] = {
            "setting": {
                key: row[key] for key in ("threshold", "min_area", "join_gap") if key in done
            },
            "pq": round(float(row["pq"]), 4),
            "sq": round(float(row["sq"]), 4),
            "rq": round(float(row["rq"]), 4),
            "tp": int(row["tp"]),
            "fp": int(row["fp"]),
            "fn": int(row["fn"]),
        }
        print(f"{name}: already swept, best {row['pq']:.4f}")
        continue

    # The whole body, not just the sweep: one run that cannot be read or
    # written must not cost the others.
    try:
        maps = load_maps(maps_dir, val_stems)
        outcome = sweep(maps, gt_df, SETTINGS, disks=disks)
        outcome.table.to_csv(f"/kaggle/working/sweep_{name}.csv", index=False)

        shared = [point for point in outcome.points if point.setting.values == SCORING]
        if shared and "pq" in results.get(name, {}):
            print(
                f"{name}: same setting -- sweep {shared[0].pq.pq:.4f}, "
                f"section 3 {results[name]['pq']:.4f}"
            )

        best = outcome.best
        retuned[name] = {
            "setting": dict(best.setting.values),
            "pq": round(best.pq.pq, 4),
            "sq": round(best.pq.sq, 4),
            "rq": round(best.pq.rq, 4),
            "tp": best.pq.tp,
            "fp": best.pq.fp,
            "fn": best.pq.fn,
        }
        print(f"{name}: best {best.pq.pq:.4f} at {best.setting}")
        del maps
    except Exception as error:
        print(f"{name}: FAILED -- {type(error).__name__}: {error}")

Path("/kaggle/working/phase6_retuned.json").write_text(json.dumps(retuned, indent=2))

## 7. What to record

Into the lab notebook, for every run and not only the winner —
the ones that did not work are what the ablation table in the final report is
made of:

- PQ, SQ, RQ, TP, FP, FN, and the fused and split counts
- best epoch and training time, noting which runs shared the session's two cards
- the commit hash this notebook cloned

Do not submit from here. The comparison is decided on fold 0 locally; folds 1
to 4 stay untouched until a configuration is frozen.